# Depression Post Classification — Solution

**Task:** Binary classification — predict if a Reddit post was written by a depressed person.  
**Metric:** F1 score  
**Data:** ~10800 train, ~973 test posts (title + body text)

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from collections import Counter
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

os.makedirs('models', exist_ok=True)
os.makedirs('submissions', exist_ok=True)

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Label distribution:\n{train["label"].value_counts()}')
print(f'Positive rate: {train["label"].mean():.3f}')

## 1. EDA

**Key findings:**
- **Class imbalance**: 19.2% positive (depressed), 80.8% negative
- **Body length** is the strongest signal: depressed posts are ~4x longer (964 vs 237 chars)
- **Empty body**: 22.9% of non-depressed posts have no body vs only 3.6% of depressed
- **Word count** correlation with label: 0.35 (strongest meta-feature)
- **Key bigrams for depression**: "kill myself", "hate myself", "suicidal thoughts", "wish could"
- **upper_ratio** negatively correlated (-0.09): depressed users write in lowercase more
- Artifacts in class 0: "pop pop", "filler filler" — noise posts

In [ ]:
train['body_filled'] = train['body'].fillna('')
train['text'] = train['title'].astype(str) + ' [SEP] ' + train['body_filled'].astype(str)

train['body_empty'] = (train['body_filled'].str.strip() == '').astype(int)
train['body_len'] = train['body_filled'].str.len()
train['word_count'] = train['text'].str.split().str.len()

print('--- Text length by class ---')
print(train.groupby('label')[['body_len', 'word_count', 'body_empty']].mean())

STOPWORDS = {'the','and','for','that','this','with','have','are','was','but','not','you','your',
             'just','like','its','from','they','been','has','had','all','can','get','got','one',
             'out','what','when','who','will','would','there','their','about','more','some','than',
             'then','into','also','very','even','know','think','want','feel','really','dont'}

def top_bigrams(texts, top_k=15):
    tokens = []
    for t in texts:
        words = [w for w in re.findall(r'\b[a-z]{3,}\b', t.lower()) if w not in STOPWORDS]
        tokens.extend([' '.join(words[i:i+2]) for i in range(len(words)-1)])
    return Counter(tokens).most_common(top_k)

print('\n--- Top bigrams DEPRESSED ---')
print(top_bigrams(train[train['label']==1]['text'].tolist()))
print('\n--- Top bigrams NOT DEPRESSED ---')
print(top_bigrams(train[train['label']==0]['text'].tolist()))

## 2. Preprocessing & Feature Engineering

**Decisions based on EDA:**
- Combine `title + [SEP] + body` — separator helps model distinguish sections
- Fill NaN body with empty string (not drop — body_empty is a signal)
- No stemming/lemmatization — TF-IDF with char n-grams handles morphology
- Keep punctuation and case info in meta-features (upper_ratio signal)
- TF-IDF word (1,2)-grams + char (2,5)-grams: captures both semantic and morphological patterns
- Meta-features: body_len, word_count, body_empty, kw_count (depression keywords) — all correlated with label

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train['body'] = train['body'].fillna('')
test['body'] = test['body'].fillna('')

train['text'] = train['title'].astype(str) + ' [SEP] ' + train['body'].astype(str)
test['text'] = test['title'].astype(str) + ' [SEP] ' + test['body'].astype(str)

DEPRESSION_KEYWORDS = [
    'kill myself', 'killing myself', 'hate myself', 'suicidal', 'suicide',
    'want to die', 'wish i was dead', 'end my life', 'no reason to live',
    'worthless', 'hopeless', 'depressed', 'depression', 'self harm',
    'cutting myself', 'overdose',
]

def meta_features(df):
    body = df['body'].astype(str)
    text = df['text'].astype(str)
    title = df['title'].astype(str)
    body_len = body.str.len()
    title_len = title.str.len()
    feats = pd.DataFrame({
        'body_len': body_len,
        'word_count': text.str.split().str.len(),
        'sent_count': text.str.count(r'[.!?]+'),
        'body_empty': (body.str.strip() == '').astype(int),
        'upper_ratio': text.apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1)),
        'question_count': text.str.count(r'\?'),
        'excl_count': text.str.count(r'!'),
        'title_len': title_len,
        'avg_word_len': text.apply(lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0),
        'body_title_ratio': body_len / (title_len + 1),
        'ellipsis_count': text.str.count(r'\.\.\.'),
        'newline_count': text.str.count(r'\n'),
        'unique_word_ratio': text.apply(
            lambda x: len(set(x.lower().split())) / max(len(x.split()), 1)
        ),
        'kw_count': text.str.lower().apply(
            lambda x: sum(1 for kw in DEPRESSION_KEYWORDS if kw in x)
        ),
    })
    return csr_matrix(feats.values.astype(float))

X_meta_train = meta_features(train)
X_meta_test = meta_features(test)

tfidf_word = TfidfVectorizer(
    ngram_range=(1, 2), max_features=150000, sublinear_tf=True,
    min_df=2, analyzer='word', token_pattern=r'\b\w+\b',
)
tfidf_char = TfidfVectorizer(
    ngram_range=(2, 5), max_features=100000, sublinear_tf=True,
    min_df=3, analyzer='char_wb',
)

all_texts = pd.concat([train['text'], test['text']])
tfidf_word.fit(all_texts)
tfidf_char.fit(all_texts)

X_train = hstack([
    tfidf_word.transform(train['text']),
    tfidf_char.transform(train['text']),
    X_meta_train
]).tocsr()
X_test = hstack([
    tfidf_word.transform(test['text']),
    tfidf_char.transform(test['text']),
    X_meta_test
]).tocsr()

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')

## 3. Modeling

| Model | OOF F1 (t=0.5) | OOF F1 (best t) | Notes |
|-------|----------------|-----------------|-------|
| LogReg baseline (lbfgs) | 0.7938 | 0.8077 (t=0.64) | TF-IDF word+char, meta |
| LinearSVC (calibrated) | 0.4991 | 0.7415 (t=0.21) | Не сошёлся |
| LogReg v2 (saga) | TBD | TBD | Больше признаков |
| LightGBM | TBD | TBD | Sparse TF-IDF |
| Ensemble | TBD | TBD | Blend лучших |

In [ ]:
y = train['label'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = LogisticRegression(
        C=5.0, max_iter=5000, solver='saga',
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    model.fit(X_tr, y_tr)

    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / 5
    fold_f1 = f1_score(y_val, (oof_preds[val_idx] >= 0.5).astype(int))
    print(f'Fold {fold+1}: F1={fold_f1:.4f}')

oof_f1 = f1_score(y, (oof_preds >= 0.5).astype(int))
print(f'\nOOF F1 (t=0.5): {oof_f1:.4f}')

best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.2, 0.8, 0.01):
    f1 = f1_score(y, (oof_preds >= t).astype(int))
    if f1 > best_f1:
        best_f1, best_t = f1, t
print(f'OOF F1 (best t={best_t:.2f}): {best_f1:.4f}')

np.save('models/logreg_v2_oof.npy', oof_preds)
np.save('models/logreg_v2_test.npy', test_preds)

## 4. Error Analysis

In [ ]:
preds = (oof_preds >= best_t).astype(int)

print(classification_report(y, preds, target_names=['not_dep', 'dep']))
print('Confusion Matrix:')
print(confusion_matrix(y, preds))

train['pred'] = preds
train['prob'] = oof_preds
train['correct'] = (train['pred'] == train['label']).astype(int)

fp = train[(train['pred'] == 1) & (train['label'] == 0)]
fn = train[(train['pred'] == 0) & (train['label'] == 1)]
print(f'\nFalse Positives: {len(fp)}, False Negatives: {len(fn)}')

print('\n--- Top False Positives ---')
for _, row in fp.sort_values('prob', ascending=False).head(5).iterrows():
    print(f'  p={row["prob"]:.3f} | {row["title"][:80]}')

print('\n--- Top False Negatives ---')
for _, row in fn.sort_values('prob', ascending=True).head(5).iterrows():
    print(f'  p={row["prob"]:.3f} | {row["title"][:80]}')

In [ ]:
train['text_len'] = train['text'].str.len()
train['len_bucket'] = pd.cut(
    train['text_len'],
    bins=[0, 100, 300, 1000, 5000, 100000],
    labels=['tiny', 'short', 'medium', 'long', 'very_long']
)
err_by_len = train.groupby('len_bucket', observed=True).apply(
    lambda g: pd.Series({
        'count': len(g),
        'error_rate': 1 - g['correct'].mean(),
        'fp_rate': ((g['pred'] == 1) & (g['label'] == 0)).mean(),
        'fn_rate': ((g['pred'] == 0) & (g['label'] == 1)).mean(),
    })
)
print('Error analysis by text length:')
print(err_by_len)

## 5. Final Submission

In [ ]:
sub = test[['id']].copy()
sub['label'] = (test_preds >= best_t).astype(int)
sub.to_csv('submissions/final_submission.csv', index=False)
print(f'Saved submissions/final_submission.csv')
print(f'Positive rate: {sub["label"].mean():.3f}')
print(sub.head(10))